# Verify Refactor: Homelab Assistant

`01_mvp_prototype.ipynb` proved the RAG concept works with logic written directly
in notebook cells. That logic has since been refactored into proper modules:

- `ingestion/ingest.py`: scrape, chunk, embed, save to `data/`
- `app/rag.py`: load data, retrieve, build prompt, call LLM

This notebook imports those modules directly and re-runs the same kind of smoke test as the MVP notebook, to confirm the refactor didn't change behavior and not a rewrite from scratch, just moving working code into reusable modules.

> **Snapshot notice**: this notebook is a point-in-time snapshot of the dev process, not a maintained test suite. `app/rag.py` and `ingestion/ingest.py` keep evolving in later commits (e.g. moving from in-memory `minsearch` to Postgres+pgvector), so running this notebook against a *later* commit may fail or behave differently than documented here. If you want this notebook to actually run, `git checkout` the commit it was added in first:
> ```bash
> git log --oneline -- notebooks/02_verify_refactor.ipynb
> git checkout <that-commit-hash>
> ```

In [1]:
import os
import sys

# notebooks/ sits one level below the project root, where the ingestion/ and
# app/ packages live
sys.path.append(os.path.abspath(".."))

from ingestion import ingest
from app import rag

## 1. Ingestion

Skips re-scraping if `data/corpus.json` already exists (from a previous run of
`ingestion/ingest.py`). Delete that file first if you want to force a fresh scrape.

In [2]:
corpus_path = os.path.join(ingest.DATA_DIR, "corpus.json")

if os.path.exists(corpus_path):
    print(f"{corpus_path} already exists, skipping re-scrape.")
else:
    ingest.run()

f:\DataTalks Certifications\homeassistant-rag\ingestion\..\data\corpus.json already exists, skipping re-scrape.


## 2. Smoke test the refactored `app.rag` module

Same two questions used in the MVP notebook. Answers should look just as
grounded/relevant now that the logic lives in `app/rag.py` instead of notebook cells.

In [3]:
for q in [
    "How do I template a sensor value in Home Assistant?",
    "How do I pair a new Zigbee device?",
]:
    result = rag.answer_question(q)
    print(f"Q: {result['question']}\n")
    print(f"A: {result['answer']}\n")
    print("Sources:", [s["url"] for s in result["sources"][:3]])
    print("-" * 100)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Q: How do I template a sensor value in Home Assistant?

A: To template a sensor value in Home Assistant, you can use the `value_template` option in the sensor configuration. This option allows you to specify a template that will be used to calculate the value of the sensor.

For example, if you want to create a sensor that displays the average temperature of two other sensors, you can use the following configuration:

```yaml
template:
  - sensor
    - name: Average Temperature
    state:
      >-
        {% set bedroom = states('sensor.bedroom_temperature') | float %}
        {% set kitchen = states('sensor.kitchen_temperature') | float %}
        {{ ((bedroom + kitchen) / 2) | round(1, default=0) }}
```

In this example, the `value_template` is used to calculate the average temperature of the `bedroom_temperature` and `kitchen_temperature` sensors. The result is then displayed as the value of the `Average Temperature` sensor.

You can also use the `value_template` option to perform m